#### 2. Tackle the Titanic dataset. A great place to start is on Kaggle. Alternatively, you can download the data from https://homl.info/titanic.tgz. This will give you two CSV files, train.csv and test.csv, which you can load using pandas.read_csv(). The goal is to train a classifier that can predict the Survived column based on the other columns.

In [92]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

##### Load Data

In [93]:
train= pd.read_csv("train.csv")
test= pd.read_csv("test.csv")

In [94]:
y_train= train['Survived']
df= pd.concat([train.drop('Survived', axis=1) ,test], ignore_index= True)

In [95]:
df.sample(5)

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
117,118,2,"Turpin, Mr. William John Robert",male,29.0,1,0,11668,21.0000,NaN,S
997,998,3,"Buckley, Mr. Daniel",male,21.0,0,0,330920,7.8208,NaN,Q
666,667,2,"Butler, Mr. Reginald Fenton",male,25.0,0,0,234686,13.0000,NaN,S
304,305,3,"Williams, Mr. Howard Hugh 'Harry'",male,NaN,0,0,A/5 2466,8.0500,NaN,S
923,924,3,"Dean, Mrs. Bertram (Eva Georgetta Light)",female,33.0,1,2,C.A. 2315,20.5750,NaN,S


##### Drop Useless Columns

In [96]:
df.drop(['Name', 'Ticket', 'Cabin'], axis=1, inplace= True) # inplace = true directly modifies datframe

In [97]:
df.sample(5)

,PassengerId,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
443,444,2,female,28.0,0,0,13.0000,S
296,297,3,male,23.5,0,0,7.2292,C
29,30,3,male,NaN,0,0,7.8958,S
1046,1047,3,male,24.0,0,0,7.5500,S
1169,1170,2,male,30.0,1,0,21.0000,S


##### Feature Engineering

In [98]:
df['FamilySize']= df['SibSp'] + df['Parch'] + 1
df= df.drop(['SibSp', 'Parch'], axis=1)

In [99]:
df.isnull().sum()

PassengerId      0
Pclass           0
Sex              0
Age            263
Fare             1
Embarked         2
FamilySize       0
dtype: int64

In [100]:
median_age= train['Age'].median()
median_fare= train['Fare'].median()
mode_embarked= train['Embarked'].mode()[0]

df['Age']= df['Age'].fillna(median_age)
df['Fare']= df['Fare'].fillna(median_fare)
df['Embarked']= df['Embarked'].fillna(mode_embarked)

In [101]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  1309 non-null   int64  
 1   Pclass       1309 non-null   int64  
 2   Sex          1309 non-null   str    
 3   Age          1309 non-null   float64
 4   Fare         1309 non-null   float64
 5   Embarked     1309 non-null   str    
 6   FamilySize   1309 non-null   int64  
dtypes: float64(2), int64(3), str(2)
memory usage: 79.0 KB


In [102]:
df['Pclass']= df['Pclass'].astype('category')
df['Sex']= df['Sex'].astype('category')
df['Age']= df['Age'].astype('int')
df['Embarked']= df['Embarked'].astype('category')

In [103]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1309 entries, 0 to 1308
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   PassengerId  1309 non-null   int64   
 1   Pclass       1309 non-null   category
 2   Sex          1309 non-null   category
 3   Age          1309 non-null   int64   
 4   Fare         1309 non-null   float64 
 5   Embarked     1309 non-null   category
 6   FamilySize   1309 non-null   int64   
dtypes: category(3), float64(1), int64(3)
memory usage: 44.9 KB


In [104]:
train_len= len(y_train)

In [105]:
X_train= df.iloc[:train_len].drop('PassengerId', axis= 1)
X_test= df.iloc[train_len:].drop('PassengerId', axis= 1)

##### Encoding Categorical Columns and scaling

In [106]:
categorical_transform= OneHotEncoder(drop='first', sparse_output=False, dtype= np.int32)
numerical_transform= StandardScaler()

##### Using ColumnTransformer

In [107]:
preprocessor= ColumnTransformer(
    transformers=[
        ('cat', categorical_transform, ['Pclass', 'Sex', 'Embarked']),
        ('num', numerical_transform, ['Age', 'Fare', 'FamilySize'])
    ]
)

##### Pipeline

In [108]:
model= make_pipeline(preprocessor, LogisticRegression())

In [109]:
scores = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
print(scores)
print("Cross Validation Score:", scores.mean())

[0.77653631 0.82022472 0.78651685 0.78089888 0.83146067]
Cross Validation Score: 0.7991274872889335


In [110]:
model.fit(X_train, y_train)
y_pred= model.predict(X_test)
y_pred

array([0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1, 1, 0, 0,
       1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1,
       1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1,
       1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1,
       0, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0,
       1, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
       0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0,

In [112]:
prediction= pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Survived': y_pred
})

prediction.to_csv('prediction.csv', index= False)